In [1]:
source_csv = '../../en.openfoodfacts.org.products.csv'

print('TP 1 CSV DF:')
print()

import platform
print(f"Python version : {platform.python_version()}")

import pandas as pd
import numpy as np
import datetime as dt

print(f"Pandas version : {pd.__version__}")
print(f"Numpy version : {np.__version__}")

TP 1 CSV DF:

Python version : 3.12.10
Pandas version : 3.0.5
Numpy version : 2.5.2


In [2]:
# Outils

from pathlib import Path
import humanize

# Time
import time

class ExecutionTime:
    _start = 0

    def __init__(self):
        self._start = 0

    def start(self):
        self._start = time.time()

    def end(self, text = ""):
        time_exec = time.time() - self._start
        print(f"Execution time {text}: {humanize.precisedelta(time_exec, minimum_unit="microseconds")}")
        print('-----------------------\n')

t = ExecutionTime()

In [3]:
print('CSV:')
print()

t.start()
file_csv = Path(source_csv)
print(f"CSV size: {humanize.naturalsize(file_csv.stat().st_size)}")
print(f"Last metadata change: {dt.datetime.fromtimestamp((file_csv.stat().st_ctime)).strftime("%Y-%m-%d %H:%M:%S")}")
t.end("(pathlib.stat())")

print('CSV SAMPLE DATA')

t.start()

df_csv = pd.read_csv(
    source_csv,
    skipinitialspace=True,
    sep="\t",
    nrows=3,
)

print(f"Dimensions : {df_csv.shape}")
print(f"→ {df_csv.shape[0]} lines, {df_csv.shape[1]} columns\n")

print('Columns name:')
columns_name = list(df_csv)
display(list(df_csv))

print('Sample data:')
transposed = np.transpose(df_csv)
with pd.option_context('display.max_rows', None):
    display(transposed)

t.end()

CSV:

CSV size: 13.0 GB
Last metadata change: 2026-08-18 08:50:54
Execution time (pathlib.stat()): 2 milliseconds and 59 microseconds
-----------------------

CSV SAMPLE DATA
Dimensions : (3, 211)
→ 3 lines, 211 columns

Columns name:


['code',
 'url',
 'creator',
 'created_t',
 'created_datetime',
 'last_modified_t',
 'last_modified_datetime',
 'last_modified_by',
 'last_updated_t',
 'last_updated_datetime',
 'product_name',
 'abbreviated_product_name',
 'generic_name',
 'quantity',
 'packaging',
 'packaging_tags',
 'packaging_en',
 'packaging_text',
 'brands',
 'brands_tags',
 'brands_en',
 'categories',
 'categories_tags',
 'categories_en',
 'origins',
 'origins_tags',
 'origins_en',
 'manufacturing_places',
 'manufacturing_places_tags',
 'labels',
 'labels_tags',
 'labels_en',
 'emb_codes',
 'emb_codes_tags',
 'first_packaging_code_geo',
 'cities',
 'cities_tags',
 'purchase_places',
 'stores',
 'countries',
 'countries_tags',
 'countries_en',
 'ingredients_text',
 'ingredients_tags',
 'ingredients_analysis_tags',
 'allergens',
 'allergens_en',
 'traces',
 'traces_tags',
 'traces_en',
 'serving_size',
 'serving_quantity',
 'no_nutrition_data',
 'additives_n',
 'additives',
 'additives_tags',
 'additives_en',
 'nu

Sample data:


,0,1,2
code,54,63,114
url,http://world-en.openfoodfacts.org/product/0000...,http://world-en.openfoodfacts.org/product/0000...,http://world-en.openfoodfacts.org/product/0000...
creator,kiliweb,kiliweb,kiliweb
created_t,1582569031,1673620307,1580066482
created_datetime,2020-02-24T18:30:31Z,2023-01-13T14:31:47Z,2020-01-26T19:21:22Z
last_modified_t,1733085204,1750061386,1751035658
last_modified_datetime,2024-12-01T20:33:24Z,2025-06-16T08:09:46Z,2025-06-27T14:47:38Z
last_modified_by,NaN,bodysupport,teolemon
last_updated_t,1740205422,1750061386,1751035658
last_updated_datetime,2025-02-22T06:23:42Z,2025-06-16T08:09:46Z,2025-06-27T14:47:38Z


Execution time : 34 milliseconds and 901 microseconds
-----------------------



In [4]:
print('CSV FULL DATA (filtrage colonnes utiles pour analyse)')

check_duplicated = ["code"]

use_cols = check_duplicated + [
    "product_name", "brands",
    "countries",
    "energy_100g", "sugars_100g", "salt_100g",
    "nutriscore_score"
]

t.start()

chunk_iterator_csv = pd.read_csv(
    source_csv,
    memory_map=True,
    skipinitialspace=True,
    sep="\t",
    chunksize=10000,
    low_memory = False,
    on_bad_lines="skip",
    usecols=use_cols
)

full_data_csv = []

for chunk in chunk_iterator_csv:
    full_data_csv.append(chunk)

df_full_csv = pd.concat(full_data_csv, ignore_index=True)

t.end('Load data')

CSV FULL DATA (filtrage colonnes utiles pour analyse)
Execution time Load data: 1 minute, 30 seconds, 517 milliseconds and 671 microseconds
-----------------------



In [5]:
print('Analyse:')

t.start()

display(df_full_csv.info())

print(f"Dimensions: {df_full_csv.shape}")
print(f"→ {df_full_csv.shape[0]} lines, {df_full_csv.shape[1]} columns")
print()

print(f"Duplication: {check_duplicated}:")
print(df_full_csv.duplicated(subset=check_duplicated).value_counts())
print()

print("Valeurs non-renseignées | Taux de remplissage par colonnes:")
missing = pd.DataFrame({
    'total_manquants': df_full_csv.isna().sum(),
    '%': 100 - (df_full_csv.isna().sum() / len(df_full_csv) * 100).round(2)
})
print(missing)
print()

# Voir https://static.openfoodfacts.org/data/data-fields.txt
print("Produits vendu en France ([countries].str.contains(fr)):")
print(df_full_csv[df_full_csv["countries"].str.contains("fr")].shape[0])
print()

print("Top 10 marques: ")
print(df_full_csv.groupby(["brands"]).size().sort_values(ascending=False)[0:10])
print()

print("Quelle part de Nutri-Score renseigné ?")
print(f"Total: {df_full_csv.shape[0]}")
print(f"Manquant: {df_full_csv["nutriscore_score"].isna().sum()}")
print(f"Nutri-Score non-renseignés: {(df_full_csv["nutriscore_score"].isna().sum() / len(df_full_csv) * 100).round(2)}%")

t.end()

Analyse:
<class 'pandas.DataFrame'>
RangeIndex: 4532767 entries, 0 to 4532766
Data columns (total 8 columns):
 #   Column            Dtype  
---  ------            -----  
 0   code              object 
 1   product_name      str    
 2   brands            str    
 3   countries         str    
 4   nutriscore_score  object 
 5   energy_100g       object 
 6   sugars_100g       float64
 7   salt_100g         float64
dtypes: float64(2), object(3), str(3)
memory usage: 434.5+ MB


None

Dimensions: (4532767, 8)
→ 4532767 lines, 8 columns

Duplication: ['code']:
False    4532701
True          66
Name: count, dtype: int64

Valeurs non-renseignées | Taux de remplissage par colonnes:
                  total_manquants       %
code                            0  100.00
product_name               335833   92.59
brands                    1682789   62.88
countries                   22828   99.50
nutriscore_score          3157914   30.33
energy_100g               2296811   49.33
sugars_100g               2388528   47.31
salt_100g                 2580977   43.06

Produits vendu en France ([countries].str.contains(fr)):
590241

Top 10 marques: 
brands
Carrefour    25641
Auchan       18107
Coop         14067
Lidl         13804
U            12286
BonÀrea      12193
Aldi         11683
Hacendado    10490
Tesco        10282
Delhaize      9741
dtype: int64

Quelle part de Nutri-Score renseigné ?
Total: 4532767
Manquant: 3157914
Nutri-Score non-renseignés: 69.67%
Execution time : 2 secon